# 10.01 Modal Sandbox

**`langchain-modal`** 이 제공하는 `ModalSandbox` 는 Modal의 서버리스 컨테이너를 Deep Agents `SandboxBackendProtocol` 구현체로 감싸 놓은 래퍼다.
에이전트가 임의 shell 커맨드·파일 I/O 를 **호스트에 닿지 않는 격리 환경**에서 수행해야 할 때 사용한다.

## 학습 목표

- `modal.Sandbox.create(...)` 로 받은 핸들을 `ModalSandbox(sandbox=...)` 에 주입하는 조립 패턴 이해
- `sandbox.execute("...")` / `read` / `write` / `upload_files` / `download_files` 5대 API 숙지
- `create_deep_agent(..., backend=ModalSandbox(...))` 연동 — 에이전트의 파일시스템·셸 도구가 전부 Modal 위로 라우팅
- **GPU · 리소스 제한** 옵션(`gpu="A10G"`, `cpu`, `memory`, `timeout`, `block_network`) 선택 기준
- 콜드스타트(수 초) vs 격리도(컨테이너) 트레이드오프

## 언제 쓰나 — Modal vs Daytona vs Runloop

| 선택 | 적합 | 특징 |
|------|------|------|
| **Modal** | 스테이트리스한 **짧은 태스크**, GPU 필요 (ML 추론·렌더), burst 병렬 | 서버리스, 밀리초 단위 과금, GPU 타입 지정 가능 |
| Daytona | **Dev Container** 이미지로 재현성 높은 개발 워크스페이스, IDE 에이전트 | `.devcontainer.json` 호환, 스냅샷·volume |
| Runloop | **세션 유지**(long-lived filesystem), 셸·브라우저 내장 | 코드 리뷰·디버깅 루프, 파일시스템 재방문 |

## 10.01.1 환경 설정

필요 패키지: `langchain-modal` (deep-agents + `modal` SDK를 끌고 온다).
`.env` 에는 `MODAL_TOKEN_ID` 와 `MODAL_TOKEN_SECRET` 가 동시에 있어야 한다. `modal setup` 로 발급한 토큰을 그대로 붙이면 된다.

```bash
uv pip install langchain-modal
modal token new   # 웹 로그인 후 .env 또는 ~/.modal.toml 에 저장
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
assert os.environ.get("MODAL_TOKEN_ID"), "MODAL_TOKEN_ID 필요"
assert os.environ.get("MODAL_TOKEN_SECRET"), "MODAL_TOKEN_SECRET 필요"

## 10.01.2 기본 사용 — 셸 하나 실행

가장 짧은 E2E 경로.

1. `modal.App()` 을 만들고 **enable_output 컨텍스트** 에서 `modal.Sandbox.create(app=..., image=...)` 로 컨테이너 한 대를 띄운다.
2. `ModalSandbox(sandbox=...)` 래퍼로 감싼 뒤 `.execute("...")` 호출.
3. 끝나면 `sandbox.terminate()` — 과금 중지.

In [ ]:
import modal
from langchain_modal import ModalSandbox

app = modal.App.lookup("lc-sandbox-hello", create_if_missing=True)
image = modal.Image.debian_slim(python_version="3.12")

with modal.enable_output():
    raw_sbx = modal.Sandbox.create(app=app, image=image, timeout=60)

sandbox = ModalSandbox(sandbox=raw_sbx)
print("sandbox id:", sandbox.id)

result = sandbox.execute("python -c 'print(1+1)'")
print("exit:", result.exit_code)
print("stdout:", result.output.strip())

raw_sbx.terminate()

## 10.01.3 `create_deep_agent(..., backend=ModalSandbox(...))` 연동

Deep Agents 하네스는 `FilesystemMiddleware` · Shell 도구 호출을 전부 `backend` 객체의 메서드로 위임한다.
`ModalSandbox` 를 backend 로 넘기면 에이전트의 `ls`, `read_file`, `write_file`, 코드 실행이 **전부 Modal 컨테이너 안**에서 일어난다.

주의: backend 는 **단일 객체**다. 한 에이전트 = 한 샌드박스. 병렬로 쓰려면 샌드박스를 여러 개 만들어 에이전트도 여러 개 띄운다.

In [ ]:
from deepagents import create_deep_agent

app = modal.App.lookup("lc-sandbox-agent", create_if_missing=True)
image = modal.Image.debian_slim(python_version="3.12").pip_install("requests")

with modal.enable_output():
    raw_sbx = modal.Sandbox.create(app=app, image=image, timeout=300, workdir="/workspace")

backend = ModalSandbox(sandbox=raw_sbx)

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[],
    system_prompt=(
        "너는 샌드박스 안에서 Python 스크립트를 작성·실행하는 엔지니어다. "
        "파일은 /workspace 아래에 만들고, 실행 결과를 짧게 요약한다."
    ),
    backend=backend,
)

res = agent.invoke({"messages": [{"role": "user", "content": "hello.py 에 'print(sum(range(100)))' 를 쓰고 실행해"}]})
print(res["messages"][-1].content)

raw_sbx.terminate()

## 10.01.4 파일 전송 · 라이프사이클

`ModalSandbox` 는 5개의 파일 API 를 제공한다.

- `write(path, content: str)` · `read(path)` — 작은 텍스트 파일 한 개
- `upload_files([(path, bytes), ...])` — 로컬 → 샌드박스 **배치 업로드**
- `download_files([path, ...])` — 샌드박스 → 로컬 **배치 다운로드**
- `edit(path, ...)` — 라인 단위 편집 (Deep Agents `edit_file` 도구 백엔드)

라이프사이클 팁:

- `modal.Sandbox.create(..., timeout=N)` — **절대 수명**. 초 단위. 0 은 무제한 (주의).
- `idle_timeout=N` — N 초 동안 exec 이 없으면 자동 종료.
- 명시 종료는 `raw_sbx.terminate()`. 예외 경로에서도 `try/finally` 는 이 노트북 정책상 쓰지 않으니, 실전 코드에서는 반드시 감싼다.

In [ ]:
app = modal.App.lookup("lc-sandbox-fs", create_if_missing=True)
image = modal.Image.debian_slim(python_version="3.12")

with modal.enable_output():
    raw_sbx = modal.Sandbox.create(app=app, image=image, timeout=120, idle_timeout=30, workdir="/workspace")

sandbox = ModalSandbox(sandbox=raw_sbx)

# 1) 텍스트 한 개 쓰기
sandbox.write("/workspace/note.txt", "hello from host")

# 2) 배치 업로드 (bytes 만)
sandbox.upload_files([
    ("/workspace/data.csv", b"a,b\n1,2\n3,4\n"),
    ("/workspace/script.py", b"import csv\nrows = list(csv.reader(open('data.csv')))\nprint(rows)\n"),
])

# 3) 실행
r = sandbox.execute("python script.py")
print("exec exit:", r.exit_code, "output:", r.output.strip())

# 4) 다운로드
downloaded = sandbox.download_files(["/workspace/note.txt"])
print("downloaded:", downloaded[0].content)

raw_sbx.terminate()

## 10.01.5 GPU · 리소스 제한

Modal의 강점은 **per-sandbox 리소스 스펙**. `modal.Sandbox.create` 에 넘기는 주요 리소스 필드:

| 필드 | 값 예 | 비고 |
|------|------|------|
| `gpu` | `"T4"`, `"L4"`, `"A10G"`, `"A100"`, `"H100"`, `"A100:4"` | 미지정 = CPU only. 접미사로 개수 지정 |
| `cpu` | `2.0` 또는 `(1.0, 4.0)` | 상한만 / (요청, 상한) 튜플 |
| `memory` | `4096` 또는 `(512, 4096)` (MiB) | 튜플은 soft/hard |
| `timeout` | `300` (초) | 절대 종료. `0` = 무제한 (권장 X) |
| `idle_timeout` | `60` | 유휴 자동 종료 |
| `block_network` | `True` | 샌드박스의 아웃바운드 네트워크 차단 — 신뢰 불가 코드에 권장 |
| `workdir` | `"/workspace"` | 기본 작업 디렉터리 |
| `image` | `modal.Image.debian_slim(...).pip_install("torch")` | 이미지 빌드 체인 (캐시됨) |

아래는 GPU + 네트워크 차단 조합 스니펫 (실제 GPU 예약이 요금·대기를 유발하므로 key-gated 이어도 한 번 더 주의).

In [ ]:
app = modal.App.lookup("lc-sandbox-gpu", create_if_missing=True)
gpu_image = (
    modal.Image.debian_slim(python_version="3.12")
    .pip_install("torch", "numpy")
)

# GPU 예약은 요금/대기가 발생하므로 주석 상태로 시그니처만 보여준다.
# with modal.enable_output():
#     raw_sbx = modal.Sandbox.create(
#         app=app,
#         image=gpu_image,
#         gpu="T4",                   # 또는 "A10G", "A100", "H100"
#         cpu=(1.0, 4.0),
#         memory=(1024, 8192),
#         timeout=600,
#         idle_timeout=120,
#         block_network=True,         # 신뢰 불가 코드
#         workdir="/workspace",
#     )
# gpu_box = ModalSandbox(sandbox=raw_sbx)
# gpu_box.execute("python -c 'import torch; print(torch.cuda.is_available())'")
# raw_sbx.terminate()
print("GPU 셀은 실제 예약을 피하기 위해 시그니처만 표기")

## 10.01.6 비용 · 지연 트레이드오프

| 축 | Modal | `ShellToolMiddleware` (host) | `PythonREPLTool` (host) |
|----|-------|------------------------------|--------------------------|
| 콜드스타트 | 수 초 (이미지 캐시 시 < 1초) | 0 | 0 |
| per-exec 지연 | ~100ms (컨테이너 내부) | ms | ms |
| 격리도 | **컨테이너** (네트워크 차단 옵션) | 호스트 프로세스 | 호스트 인프로세스 |
| 파일시스템 | 샌드박스 로컬 (기본 휘발) | 호스트 | 호스트 |
| 비용 | **실행 시간 · 리소스 기준 과금** | 0 | 0 |
| 적합 | 신뢰 불가 코드, GPU, 병렬 burst | 내부 CI · 개발자 로컬 | 빠른 프로토타입 |

설계 힌트:

- **같은 이미지를 재사용**하면 콜드스타트가 대부분 사라진다 (Modal이 이미지 레이어를 캐시).
- **대화 한 턴당 샌드박스 한 대** 보다 **세션 내내 공유** 가 효율적. 세션이 끝나는 시점에 `terminate()`.
- 장시간 격리 + 파일시스템 유지가 필요하면 Modal 대신 **Runloop** (10.03) 가 더 자연스럽다.

## 체크리스트

- [ ] `.env` 에 `MODAL_TOKEN_ID` + `MODAL_TOKEN_SECRET` 둘 다 존재
- [ ] `modal.Sandbox.create(..., timeout=...)` 의 절대 수명 지정 — 0(무제한)은 금지
- [ ] 신뢰 불가 코드는 `block_network=True`
- [ ] 세션 종료 시 `raw_sbx.terminate()` 호출 (과금 중지)
- [ ] `create_deep_agent(..., backend=ModalSandbox(sandbox=raw_sbx))` — backend 는 단일 객체
- [ ] GPU 는 `gpu="T4"` 같은 문자열, 개수는 `"A100:4"` 형식

## 다음

- `04_deepagents/10_sandboxes_and_acp.ipynb` — Deep Agents × 샌드박스 실전(프로젝트 형식)
- `08_integration/10_sandboxes/02_daytona.ipynb` — Dev Container 계열
- `08_integration/10_sandboxes/03_runloop.ipynb` — 세션형 샌드박스

## 참고

- `langchain-modal` PyPI: https://pypi.org/project/langchain-modal/
- Modal Sandboxes 문서: https://modal.com/docs/guide/sandbox
- Deep Agents backend protocol: `deepagents.backends.protocol.BackendProtocol`